In [0]:
import json, random, time, uuid
from datetime import datetime, timezone
 
CAT, SCH = "capstone_five", "bronze"                         # was "project"
OUT = "/Volumes/capstone_five/project/landing/vitals/"       # unchanged (volume stays in project)
 
# Pick the same 30 real patients every run (orderBy makes it repeatable)
ids = [r[0] for r in spark.table(f"{CAT}.{SCH}.dim_patients")
                          .select("Id").orderBy("Id").limit(30).collect()]
print("Patients selected:", len(ids))
 
ROUNDS = 12          # 12 rounds x 5 sec = 1 minute of data
for i in range(ROUNDS):
    rows = []
    for p in ids:
        hr = round(random.gauss(80, 10))              # normal heart rate
        if random.random() < 0.05:
            hr = random.randint(115, 160)              # ~5% tachycardia
        rows.append({
            "patient_id": p,
            "ts": datetime.now(timezone.utc).replace(tzinfo=None).isoformat(),
            "heart_rate": hr,
            "glucose": round(random.gauss(110, 25)),
            "spo2": round(random.gauss(97, 1.5)),
        })
    dbutils.fs.put(f"{OUT}{uuid.uuid4()}.json", "\n".join(json.dumps(r) for r in rows))
    print(f"Batch {i+1}/{ROUNDS} written ({len(rows)} readings)")
    time.sleep(5)